# Phase 9 — Production Recommendation Serving & API Layer

## 1. Title & Objective
This notebook demonstrates the **Production Recommendation Serving & API Layer** (`src/api.py`), providing a RESTful API wrapper around the completed Phase 1–8 recommendation system using **FastAPI**, **Pydantic**, and **TestClient**.

### 2. Why an API Layer?
Machine learning models deliver business value when integrated into application workflows. A REST API decouples model inference logic from downstream client applications (web frontends, mobile apps, microservices) via standard HTTP requests and JSON payloads.

### 3. Existing Recommendation Architecture
Phase 9 serves the complete pipeline without modifying core algorithms:
- **Content Engine (Phase 3 & 4)**: TF-IDF feature space + weighted user preference profile vector $\mathbf{u}$.
- **Collaborative Filtering (Phase 6)**: Item-Item Cosine Similarity matrix $S$ over MovieLens interaction logs.
- **Hybrid Fusion Engine (Phase 7)**: Candidate pool union ($N_{\text{cand}}=100$), Min-Max score normalization, and weighted linear fusion $\text{score}_{\text{hybrid}} = \alpha \cdot \text{norm\_content} + (1 - \alpha) \cdot \text{norm\_cf}$.
- **Explainability Layer (Phase 8)**: Decomposes recommendation evidence into factual metadata overlaps, collaborative item contributions, and hybrid weights.

In [ ]:
import os
import sys
from pathlib import Path
from fastapi.testclient import TestClient

# Robust project root resolution for notebook environment or CLI execution
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.api import app, init_models

print("Initializing API models and FastAPI TestClient...")
models = init_models()
client = TestClient(app)
print("FastAPI app & TestClient initialized successfully.")

### 5. Health Check Endpoint
We test `GET /health` to verify service readiness and dataset status.

In [ ]:
res = client.get("/health")
print(f"GET /health Status: {res.status_code}")
print("Response Payload:")
import json
print(json.dumps(res.json(), indent=2))

### 6. Content Recommendation API
We query `GET /recommend/content?title=Avatar&top_n=5` to retrieve content-similar movies based on TF-IDF metadata vectors.

In [ ]:
res = client.get("/recommend/content?title=Avatar&top_n=5")
print(f"GET /recommend/content Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

### 7. Personalized Recommendation API
We send `POST /recommend/personalized` with a user rating history payload.

In [ ]:
payload = {
    "history": [
        {"title": "Avatar", "rating": 5},
        {"title": "Aliens", "rating": 5},
        {"title": "The Dark Knight", "rating": 4},
        {"title": "Titanic", "rating": 1}
    ],
    "top_n": 5
}
res = client.post("/recommend/personalized", json=payload)
print(f"POST /recommend/personalized Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

### 8. Hybrid Recommendation API
We query `POST /recommend/hybrid` fusing content and collaborative filtering over the union candidate pool.

In [ ]:
payload = {
    "user_id": 1,
    "alpha": 0.5,
    "top_n": 5
}
res = client.post("/recommend/hybrid", json=payload)
print(f"POST /recommend/hybrid Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

### 9. Explainable Recommendation API
We query `POST /recommend/explain` to retrieve factual content, collaborative, and score decomposition evidence for a recommendation.

In [ ]:
payload = {
    "user_id": 1,
    "target_movie_id": 2918,
    "alpha": 0.5
}
res = client.post("/recommend/explain", json=payload)
print(f"POST /recommend/explain Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

### 10. Request Validation & Error Handling
We demonstrate robust Pydantic validation and error responses (400 Bad Request, 404 Not Found, 422 Unprocessable Entity).

In [ ]:
print("--- Test 1: Unknown Movie Title (Expect 404) ---")
res = client.get("/recommend/content?title=UnknownMovieXYZ&top_n=5")
print(f"Status: {res.status_code} | Payload: {res.json()}")

print("\n--- Test 2: Invalid Alpha > 1.0 (Expect 422) ---")
res = client.post("/recommend/hybrid", json={"user_id": 1, "alpha": 1.5, "top_n": 5})
print(f"Status: {res.status_code} | Payload: {res.json()}")

print("\n--- Test 3: Duplicate Movie Titles in History (Expect 422) ---")
res = client.post("/recommend/personalized", json={"history": [{"title": "Avatar", "rating": 5}, {"title": "Avatar", "rating": 4}], "top_n": 5})
print(f"Status: {res.status_code} | Payload: {res.json()}")

### 11. API Limitations & Production Considerations
1. **Serving Demonstration Scope**: This API layer provides in-process serving demonstration via FastAPI. It does **NOT** include persistent user databases, user authentication/JWT, Redis caching, microservices, Docker, or cloud deployment.
2. **Single-Node In-Memory Footprint**: Models and feature matrices are retained in application memory (`ModelContainer` singleton). Scalability across multi-node clusters requires decoupled model registries or external inference workers.
3. **Dataset Dependency**: Raw MovieLens interaction files remain local and git-ignored. Title identity alignment between MovieLens and TMDB 5000 is fixed at 2,812 mapped movies (28.86% coverage).

### 12. Conclusion
Phase 9 completes the production recommendation serving layer, successfully transitioning the Personalized Movie Recommendation System from data preprocessing, vectorization, personalization, evaluation, collaborative filtering, hybrid fusion, and explainability into a structured, production-grade REST API.